In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import butter, lfilter, iirnotch
import random
from dataclasses import dataclass
from pathlib import Path
from omegaconf import OmegaConf

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import cohen_kappa_score

from graph_attention.models.eeg_encoder import EEGEncoder
from graph_attention.training.trainer import GraphAttentionTrainer
from trainer_tools.all import (
    ProgressBarHook,
    CheckpointHook,
    MetricsHook,
)
from trainer_tools.hooks.metrics import Loss, Accuracy


In [ ]:
TARGET_N_TIMES = 1125  # 4.5s * 250Hz
SFREQ = 250.0


def bandpass_filter(data, lowcut=0.5, highcut=100.0, fs=SFREQ, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return lfilter(b, a, data, axis=0)


def notch_filter(data, freq=50.0, fs=SFREQ, q=30.0):
    b, a = iirnotch(freq / (0.5 * fs), q)
    return lfilter(b, a, data, axis=0)


def load_data_from_mat(file_path):
    mat = scipy.io.loadmat(file_path, struct_as_record=False, squeeze_me=True)
    all_x = []
    all_y = []

    mi_runs = mat['data'][-6:] 
    
    for run in mi_runs:
        fs = float(run.fs)
        # 1) Фильтрация (0.5-100 Гц) + notch 50 Гц
        x_full = bandpass_filter(run.X[:, :22], fs=fs)
        x_full = notch_filter(x_full, freq=50.0, fs=fs)

        # 2) Нарезка на триалы
        for j, start_sample in enumerate(run.trial):
            end_sample = start_sample + TARGET_N_TIMES
            trial_data = x_full[start_sample:end_sample, :]  # (1125, 22)

            # Проверка на артефакты
            if hasattr(run, 'artifacts') and run.artifacts[j] != 0:
                continue

            all_x.append(trial_data)
            all_y.append(run.y[j] - 1)  # 1-4 -> 0-3

    return np.array(all_x), np.array(all_y)


def standardize_train_test(x_train, x_test):
    # x: (n, t, ch) -> нормируем по тренировке для каждого канала
    mean = x_train.mean(axis=(0, 1), keepdims=True)
    std = x_train.std(axis=(0, 1), keepdims=True) + 1e-6
    return (x_train - mean) / std, (x_test - mean) / std


In [3]:
class EEGDataset(Dataset):
    def __init__(self, x, y):
        # Модель EEGEncoder ожидает (B, 1, T, C) -> (B, 1, 1125, 22)
        self.x = torch.tensor(x, dtype=torch.float32).unsqueeze(1)
        self.y = torch.tensor(y, dtype=torch.long)
        
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

In [ ]:
def seed_all(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


@dataclass
class TrainConfig:
    epochs: int = 500
    batch_size: int = 64
    lr: float = 1e-3
    label_smoothing: float = 0.1
    dropout: float = 0.3
    weight_decay: float = 0.5  # только для Linear
    spectral_loss_lambda: float = 0.01
    checkpoint_every_epochs: int = 10


def _param_groups_linear_decay(model: nn.Module, weight_decay: float):
    decay, no_decay = [], []
    for module in model.modules():
        for name, param in module.named_parameters(recurse=False):
            if not param.requires_grad:
                continue
            if isinstance(module, nn.Linear):
                decay.append(param)
            else:
                no_decay.append(param)
    return [
        {"params": decay, "weight_decay": weight_decay},
        {"params": no_decay, "weight_decay": 0.0},
    ]


def evaluate(model, loader):
    model.eval()
    all_preds = []
    all_targets = []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            preds = logits.argmax(dim=1)
            all_preds.append(preds.cpu().numpy())
            all_targets.append(y.cpu().numpy())

    y_true = np.concatenate(all_targets)
    y_pred = np.concatenate(all_preds)
    acc = (y_true == y_pred).mean() * 100.0
    kappa = cohen_kappa_score(y_true, y_pred) * 100.0
    return acc, kappa


def run_subject_agfl(data_path: str, subject_id: int, cfg: TrainConfig, seeds: list[int]):
    x_train, y_train = load_data_from_mat(f"{data_path}/A{subject_id:02d}T.mat")
    x_test, y_test = load_data_from_mat(f"{data_path}/A{subject_id:02d}E.mat")

    x_train, x_test = standardize_train_test(x_train, x_test)

    train_loader = DataLoader(EEGDataset(x_train, y_train), batch_size=cfg.batch_size, shuffle=True)
    test_loader = DataLoader(EEGDataset(x_test, y_test), batch_size=cfg.batch_size, shuffle=False)

    steps_per_epoch = len(train_loader)
    save_every_steps = max(steps_per_epoch * cfg.checkpoint_every_epochs, steps_per_epoch)

    seed_metrics = []

    for seed in seeds:
        seed_all(seed)
        
        # Инициализация модели, которая по умолчанию использует AGFL
        model = EEGEncoder(eeg_channels=22, num_classes=4, dropout=cfg.dropout).to(DEVICE)
        
        optimizer = torch.optim.Adam(
            _param_groups_linear_decay(model, cfg.weight_decay),
            lr=cfg.lr,
        )

        trainer = GraphAttentionTrainer(
            model=model,
            train_dl=train_loader,
            valid_dl=test_loader,
            optim=optimizer,
            loss_func=nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing),
            epochs=cfg.epochs,
            hooks=[
                ProgressBarHook(),
                CheckpointHook(
                    f"outputs/eeg_bci2a/agfl_subject_{subject_id}/seed_{seed}/checkpoints",
                    save_every_steps=save_every_steps,
                ),
                # verbose=False чтобы не спамить в консоль на 500 эпохах
                MetricsHook(verbose=False, metrics=[Loss(), Accuracy()]), 
            ],
            config=OmegaConf.create({"training": {"spectral_loss_lambda": cfg.spectral_loss_lambda}}),
        )

        trainer.fit()
        acc, kappa = evaluate(model, test_loader)
        seed_metrics.append({"seed": seed, "acc": acc, "kappa": kappa})

    return seed_metrics


def aggregate_metrics(seed_metrics):
    accs = np.array([m["acc"] for m in seed_metrics], dtype=np.float32)
    kappas = np.array([m["kappa"] for m in seed_metrics], dtype=np.float32)
    return {
        "acc_mean": float(accs.mean()),
        "acc_std": float(accs.std(ddof=0)),
        "kappa_mean": float(kappas.mean()),
        "kappa_std": float(kappas.std(ddof=0)),
    }


Device: cpu


In [ ]:
DATA_PATH = "./data/BCICIV_2a_mat"
SUBJECTS = list(range(1, 10))
SEEDS = [0, 1, 2, 3, 4] 

cfg = TrainConfig()
results = {}

print("\n=== Starting AGFL Evaluation ===")
for subject_id in SUBJECTS:
    print(f"  Running Subject {subject_id:02d}...")
    seed_metrics = run_subject_agfl(DATA_PATH, subject_id, cfg, SEEDS)
    results[subject_id] = aggregate_metrics(seed_metrics)
    print(f"  Subject {subject_id:02d} Results: {results[subject_id]}")

# Усреднение по всем субъектам
our_agfl_mean_acc = np.mean([results[s]["acc_mean"] for s in SUBJECTS])
our_agfl_mean_kappa = np.mean([results[s]["kappa_mean"] for s in SUBJECTS])

print("\n=== FINAL AGFL RESULTS ===")
print(f"Mean Accuracy across 9 subjects: {our_agfl_mean_acc:.2f}%")
print(f"Mean Kappa across 9 subjects: {our_agfl_mean_kappa:.2f}%")

# Данные из статьи для сравнения
paper_baselines = {
    "ATCNet": 84.1,        # [cite: 344]
    "TCNetFusion": 74.6,   # [cite: 344]
    "EEGTCNet": 67.4,      # [cite: 344]
    "EEGEncoder (Base)": 74.48 # Данные из Table 3 [cite: 363]
}

labels = list(paper_baselines.keys()) + ["Our AGFL"]
values = list(paper_baselines.values()) + [our_agfl_mean_acc]
colors = ['gray', 'gray', 'gray', 'blue', 'orange']

plt.figure(figsize=(10, 6))
bars = plt.bar(labels, values, color=colors)
plt.ylabel('Mean Accuracy (%)')
plt.title('Performance Comparison on BCI-2a Dataset (All Subjects)')
plt.ylim(50, 100)

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.5, f"{yval:.1f}%", ha='center', va='bottom')

plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("outputs/final_comparison.png")
plt.show()

Subject 01


Epoch:   0%|          | 0/500 [00:00<?, ?it/s]

Epoch 1/500 [Train]:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch 1/500 [Valid]:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch 2/500 [Train]:   0%|          | 0/5 [00:00<?, ?it/s]

...

Epoch 499/500 [Valid]:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch 500/500 [Train]:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch 500/500 [Valid]:   0%|          | 0/5 [00:00<?, ?it/s]

{'acc_mean': 24.908424377441406, 'acc_std': 0.0, 'kappa_mean': 0.0, 'kappa_std': 0.0}
Subject 04


IndexError: index 7 is out of bounds for axis 0 with size 7